# UK Government Legislation Summary Tool

## Web Scraper

Below is a method for scraping readable policy text from https://www.legislation.gov.uk/ simply find the link to a compatable policy document and provide it to the tool.

An example policy you can use is https://www.legislation.gov.uk/ukpga/2025/18/contents

### Code

In [ ]:
pip install -r -q requirements.txt

In [ ]:
import feedparser
from bs4 import BeautifulSoup
#import pandas as pd
import requests
import easygui
import logging
import re
import unittest
import time

In [ ]:
class gov_uk_scraper:

    def input_url():

        # Prompt the user to enter the URL of the legislation's Table of Contents page
        
        myvar = easygui.enterbox("Please enter the URL to the Table of Contents page of the legislation you want to assess:", "UK Legislation NLP Comparison Tool")
        if myvar is None:
            return None

        # Parse the RSS feed of the legislation's Table of Contents page
        # Includes error handling to catch any exceptions that may occur during parsing
        
        else:
            try:
                feed = feedparser.parse(myvar + "/data.feed")
                # Check if the feed has entries and get the first entry's link
                url = feed.entries[0].link
                return url
            # If no entires are found call error handling function
            except Exception:
                return gov_uk_scraper.input_error("notfound")

    def input_error(error):

        # Recall input_url() if the user selects "Retry" from the error message box
        def retry_input(option):
            if option == "Retry": 
                # Respecting https://www.legislation.gov.uk/robots.txt which requests a 5 second delay between requests to the site
                start = time.time()
                while time.time() - start < 5:
                    easygui.msgbox("Please wait 5 seconds between requests", "UK Legislation NLP Comparison Tool")
                return gov_uk_scraper.input_url()
            return None

        # Display error when no url provided or no legislation found in data feed
        if error == "notfound":
            option = easygui.buttonbox("Unable to find a link to that legislation", "UK Legislation NLP Comparison Tool", ["Retry", "Exit"])
            return retry_input(option)
        # Display error when the url provided is not a valid legislation.gov.uk page
        elif error == "wrongsite":
            option = easygui.buttonbox("The URL provided does not appear to be a valid legislation.gov.uk page. Please check the URL and try again.", "UK Legislation NLP Comparison Tool", ["Retry", "Exit"])
            return retry_input(option)
        # Display error when the legislation is only available as a PDF and cannot be processed by the tool
        elif error == "pdfpage":
            option = easygui.buttonbox("The legislation provided is only available as a PDF and unforunately incompatible with this tool.", "UK Legislation NLP Comparison Tool", ["Retry", "Exit"])
            return retry_input(option)
        # Display error when the legislation does not have an Explanatory Memorandum and cannot be compared
        elif error == "noemlink":
            option = easygui.buttonbox("The legislation provided does not appear to have an Explanatory Memorandum. No Comparison can be made", "UK Legislation NLP Comparison Tool", ["Retry", "Exit"])
            return retry_input(option)

    def get_legislation_text(url):
        # Calls error handling function if legislation is only available as PDF
        if url.endswith(".pdf"):
            return gov_uk_scraper.input_error("pdfpage")
        
        #Initial web scraping
        else:
            response = requests.get(url)
            soup = BeautifulSoup(response.content, 'html.parser')

            # Final validation check to ensure website is legislation.gov.uk
            site_check = "legislation.gov.uk" in soup.find("div", id="header").get_text()
            # Checking for the presence of an Explanatory Memorandum link on the legislation page
            em_check = soup.find("li", id=["legEmLink","legEnLink"])
            if not site_check:
                return gov_uk_scraper.input_error("wrongsite")
            elif not em_check:
                return gov_uk_scraper.input_error("noemlink")
            # Text of legislation is contained within a div with class "LegSnippet"
            else:
                legislation_text = soup.find("div", class_="LegSnippet")
                return legislation_text

    def prettify_text(content):
        lines = []
        for el in content.find_all(["h1", "h2", "h3", "h4", "p", "li", "td", "div"]):
            # skip elements that just contain other elements we'll already visit —
            # otherwise you'd get duplicated text
            if el.find(["p", "li", "td", "div", "h1", "h2", "h3", "h4"]):
                continue
            text = el.get_text(" ", strip=True)
            if text:
                lines.append(text)
        readable = "\n\n".join(lines)
        return readable

    def main():
        url = gov_uk_scraper.input_url()
        # Exiting Program if no valid URL is provided
        if url is None:
            return "No valid URL provided. Exiting program."
        else:
            legislation_text = gov_uk_scraper.get_legislation_text(url)
            readable_text = gov_uk_scraper.prettify_text(legislation_text)
            print(readable_text)
            return readable_text

In [ ]:
text = gov_uk_scraper.main()

### Unit Testing

In [ ]:
# To be included before Summative!

# Early Testing Cells

In [ ]:
myvar = easygui.enterbox("test", "test")
print(myvar)

In [ ]:
pip install sumy

In [ ]:
%pip install sumy

%pip install lxml_html_clean

%pip install requests beautifulsoup4

%pip install numpy



import requests # Import requests library

from bs4 import BeautifulSoup # Add BeautifulSoup for HTML parsing

from sumy.parsers.plaintext import PlaintextParser

from sumy.nlp.tokenizers import Tokenizer

from sumy.summarizers.luhn import LuhnSummarizer # Import LuhnSummarizer

from sumy.summarizers.lex_rank import LexRankSummarizer # Import LexRankSummarizer

from sumy.summarizers.lsa import LsaSummarizer # Import LsaSummarizer

from sumy.nlp.stemmers import Stemmer

from sumy.utils import get_stop_words

import nltk 

nltk.download('punkt_tab')

In [ ]:
# LSA Extractive Summarization Example



def lsa_summarize(input_data, sentence_count=2, input_type="text"):


#    Summarize text using the LSA algorithm.



 #   Args:

 #       input_data (str): The input text or URL to summarize.
#
 #       sentence_count (int): Number of sentences for the summary.
#
 #       input_type (str): Type of input - “text” or “url”.
#


 #   Returns: list: Summary sentences.

    if input_type == "url":

        response = requests.get(input_data)

        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'html.parser') # Parse HTML content

        text = soup.get_text(separator=' ') # Extract plain text

    else: text = input_data



    # Parse the input text

    parser = PlaintextParser.from_string(text, Tokenizer("english"))



    # Initialize LSA summarizer with stemmer

    summarizer = LsaSummarizer(Stemmer("english"))

    summarizer.stop_words = get_stop_words("english")



    # Generate summary

    summary = summarizer(parser.document, sentence_count)

    return summary



# Test with sample text

sample_text = """

Text summarization is an important area of natural language processing (NLP) that focuses on condensing large amounts of text into shorter, coherent summaries. Modern approaches can identify the main ideas in a document and present them with minimal human involvement. Extractive methods select representative sentences directly from the source text, while abstractive methods generate new phrasing based on the original meaning. These techniques are increasingly used in information retrieval, research analysis, and other applications where quick understanding of text is essential.

"""



# Summarize plain text

summary = lsa_summarize(text, 20, input_type="text")

print("Summary from text:")

for sentence in summary:

    print(sentence)

In [ ]:
def get_legislation_text(url):
    if url.endswith(".pdf"):
        return input_error("pdfpage")
    else:
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        #Final validation check to ensure website is legislation.gov.uk
        site_check = "legislation.gov.uk" in soup.find("div", id="header").get_text()
        em_check = soup.find("li", id="legEmLink")
        if not site_check:
            return input_error("wrongsite")
        elif not em_check:
            return input_error("noemlink")
        else:
            legislation_text = soup.find("div", class_="LegSnippet")
        return legislation_text

In [ ]:
def prettify_text(content):
    lines = []
    for el in content.find_all(["h1", "h2", "h3", "h4", "p", "li", "td", "div"]):
        # skip elements that just contain other elements we'll already visit —
        # otherwise you'd get duplicated text
        if el.find(["p", "li", "td", "div", "h1", "h2", "h3", "h4"]):
            continue
        text = el.get_text(" ", strip=True)
        if text:
            lines.append(text)
    readable = "\n\n".join(lines)
    return readable

In [ ]:
def main():
    url = input_url()
    legislation_text = get_legislation_text(url)
    readable_text = prettify_text(legislation_text)
    print(readable_text)

In [ ]:
main()